In [15]:
import re
import unicodedata
import pandas as pd

# ========= CẤU HÌNH =========

# Các từ cần loại bỏ (không phân biệt hoa/thường)
# Lưu ý: KHÔNG đưa "sport" vào đây để còn dùng làm "filler" khi tên quá ngắn
remove_words = [
    "thời trang", "nhựa đúc", "màu", "size",
    "đi học", "thể thao", "đi mưa", "thun",
    "cao cấp", "full", "sơn", "tam giác", "đế",
    "quai ngang", "thêu", "bánh mì", "bít mũi gắn",
    "học sinh", "sneaker", "logo", "họa tiết", r"\bin\b",
    "in chữ", "tem", "tem hoạ tiết", "nhựa", "khóa xoay", "đế cao", "khoen gài", "quai thun"
]

# Mapping thay thế đặc biệt (ưu tiên cụm dài trước)
replace_map = {
    "quai hậu xoay": "khóa xoay",
    "nam nữ": "unisex",
    "Quai hậu": "Sandal",
    "Little": "Li"
}

# Bảng màu (pattern regex -> tên màu chuẩn để hiển thị)
# Đặt cụm ghép trước màu đơn để ưu tiên match đúng
color_map = [
    (r"\bxám nhạt\b", "xám nhạt"),
    (r"\bxanh nhạt\b", "xanh nhạt"),
    (r"\bnâu nhạt\b", "nâu nhạt"),
    (r"\bđỏ nhạt\b", "đỏ nhạt"),
    (r"\bvàng nhạt\b", "vàng nhạt"),

    (r"\bxanh đậm\b", "xanh đậm"),
    (r"\bđỏ đậm\b", "đỏ đậm"),

    (r"\bxanh dương\b", "xanh dương"),
    (r"\bxanh trời\b",  "xanh trời"),
    (r"\bxanh biển\b",  "xanh dương"),
    (r"\bxanh navy\b",  "navy"),
    (r"\bnavy\b",       "navy"),
    (r"\bxanh rêu\b",   "xanh rêu"),
    (r"\bxanh lá\b",    "xanh lá"),
    (r"\bxanh ngọc\b",  "xanh ngọc"),
    (r"\bmint\b",       "mint"),
    (r"\bđỏ\b",         "đỏ"),
    (r"\bhồng\b",       "hồng"),
    (r"\bđen\b",        "đen"),
    (r"\btrắng\b",      "trắng"),
    (r"\bnâu\b",        "nâu"),
    (r"\bbe\b",         "be"),
    (r"\bkem\b",        "kem"),
    (r"\bvàng\b",       "vàng"),
    (r"\bcam\b",        "cam"),
    (r"\btím\b",        "tím"),
    (r"\bxám\b",        "xám"),
    (r"\bghi\b",        "ghi"),
    (r"\bbạc\b",        "bạc"),
    # KHÔNG đưa (r"\bnhạt\b","nhạt") để tránh đảo "nâu nhạt" -> "nhạt nâu"
]

# Regex size: K3/K10, M4/M12, hoặc 2 chữ số (36–49), có thể kèm hậu tố W/A...
SIZE_RE = re.compile(
    r"\b(?:K\d{1,2}|M\d{1,2}|(?:3[0-9]|4[0-9])[A-Z]?W?)\b",
    flags=re.IGNORECASE
)
SPORT_RE = re.compile(r"\bsport\b", flags=re.IGNORECASE)

# Nhận dạng (giới tính) dạng ngoặc và chuẩn hoá
GENDER_PAREN_RE = re.compile(
    r"\((?:nam|nữ|nu|female|male|unisex|kid|kids?|bé(?:\s*(?:trai|gái))?)\)",
    flags=re.IGNORECASE
)

# ========= HELPERS =========

def _normalize_spaces(s: str) -> str:
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u00A0", " ")
    return " ".join(s.split())

def _ci_sub(text: str, pat: str, repl: str) -> str:
    if pat.startswith(r"\b") or pat.endswith(r"\b") or ("\\" in pat):
        return re.sub(pat, repl, text, flags=re.IGNORECASE)
    return re.sub(rf"\b{re.escape(pat)}\b", repl, text, flags=re.IGNORECASE)

def _remove_words_ci(text: str, words: list[str]) -> str:
    s = text
    for w in words:
        s = _ci_sub(s, w, " ")
    return " ".join(s.split())

def _find_colors_in_order(text: str, keep: int = 2) -> list[str]:
    hits = []
    for pat, canonical in color_map:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            hits.append((m.start(), canonical))
    if not hits:
        return []
    hits.sort(key=lambda x: x[0])
    seen, ordered = set(), []
    for _, c in hits:
        if c not in seen:
            seen.add(c)
            ordered.append(c)
        if len(ordered) >= keep:
            break
    return ordered

def _remove_all_colors(text: str) -> str:
    s = text
    for pat, _ in color_map:
        s = re.sub(pat, " ", s, flags=re.IGNORECASE)
    return " ".join(s.split())

def _safe_truncate_words(s: str, maxlen: int = 36) -> str:
    if len(s) <= maxlen:
        return s
    cut = s[:maxlen+1].rstrip()
    last_space = cut.rfind(" ")
    if last_space == -1:
        return s[:maxlen].rstrip()
    return cut[:last_space].rstrip()

def _extract_last_size(s: str):
    sizes = list(SIZE_RE.finditer(s))
    if not sizes:
        return s, None
    token = sizes[-1].group(0)
    base  = SIZE_RE.sub(" ", s)
    return " ".join(base.split()), token

def _extract_gender_token(s: str):
    """
    Tìm token (giới tính) trong ngoặc. Lấy cái cuối cùng nếu có nhiều.
    Chuẩn hoá về (nam)/(nữ)/(unisex)/(kid)…
    """
    matches = list(GENDER_PAREN_RE.finditer(s))
    if not matches:
        return s, None
    last = matches[-1].group(0)  # ví dụ "(nữ)"
    inner = last.strip("()").strip().lower()
    # Chuẩn hoá
    if inner in ("nu", "nữ", "nữ"):
        inner_std = "nữ"
    elif inner in ("nam", "male"):
        inner_std = "nam"
    elif inner in ("unisex",):
        inner_std = "unisex"
    elif inner in ("kid", "kids"):
        inner_std = "kid"
    elif inner.startswith("bé"):
        inner_std = "kid"
    else:
        inner_std = inner
    token = f"({inner_std})"
    # Xoá tất cả token giới tính khỏi base
    base = GENDER_PAREN_RE.sub(" ", s)
    return " ".join(base.split()), token

def _build_name(base: str, colors: list[str], size_token: str | None, gender_token: str | None) -> str:
    parts = [base]
    if colors:
        parts.append(" ".join(colors))
    if size_token:
        parts.append(size_token)
    if gender_token:
        parts.append(gender_token)
    return " ".join([p for p in parts if p]).strip()

# ========= HÀM CHÍNH =========

def simplify_name(x, maxlen: int = 36, minlen: int = 30) -> str:
    """
    Rút gọn tên về 30–36 ký tự:
    - Giữ tối đa 2 màu theo thứ tự xuất hiện (ví dụ: 'be đen')
    - Luôn giữ size; nếu có (giới tính) thì đặt SAU size: '... 39 (nữ)'
    - Nếu <30 ký tự, có thể chèn/giữ 'sport' (nếu có trong tên gốc) để đủ độ dài.
    - Nếu phát hiện (nữ) ở cuối → loại 'unisex' khỏi phần base.
    """
    if pd.isna(x):
        return x
    original = str(x)
    s = _normalize_spaces(original)

    # 1) Mapping đặc biệt (ưu tiên cụm dài trước)
    for k, v in sorted(replace_map.items(), key=lambda kv: -len(kv[0])):
        if k.lower() == "little":
            s = re.sub(r"\bLittle\b", v, s, flags=re.IGNORECASE)
        else:
            s = _ci_sub(s, k, v)
    s = _normalize_spaces(s)

    # 2) Bắt (giới tính) trong ngoặc, chuẩn hoá và loại khỏi base
    s, gender_token = _extract_gender_token(s)

    # 2.1) Nếu đuôi là (nữ) → bỏ mọi "unisex" khỏi phần base
    if gender_token and gender_token.lower() == "(nữ)":
        s = re.sub(r"\bunisex\b", " ", s, flags=re.IGNORECASE)
        s = _normalize_spaces(s)

    # 3) Cờ 'sport' (để có thể dùng làm filler nếu tên ngắn)
    had_sport = bool(SPORT_RE.search(s))

    # 4) Tách size (giữ size cuối)
    s, size_token = _extract_last_size(s)

    # 5) Bắt tối đa 2 màu theo thứ tự xuất hiện, rồi xoá hết màu khỏi base
    colors = _find_colors_in_order(s, keep=2)
    if colors:
        s = _remove_all_colors(s)

    # 6) Xoá từ thừa
    s = _remove_words_ci(s, remove_words)

    # 7) Lắp ghép (base + màu + size + (giới tính)), ưu tiên chừa chỗ cho tail
    tail_parts = []
    if colors:
        tail_parts.append(" ".join(colors))
    if size_token:
        tail_parts.append(size_token)
    if gender_token:
        tail_parts.append(gender_token)
    tail = " ".join(tail_parts).strip()

    if tail:
        reserved = len(tail) + 1  # chừa 1 khoảng trắng nối
        base_budget = maxlen - reserved
        base = _safe_truncate_words(s, maxlen=base_budget) if base_budget > 0 else ""
        candidate = _build_name(base, colors, size_token, gender_token)
    else:
        base = _safe_truncate_words(s, maxlen=maxlen)
        candidate = base

    # 8) Nếu vượt maxlen → giảm dần màu (vẫn giữ size + giới tính)
    if len(candidate) > maxlen and colors and (size_token or gender_token):
        candidate = _build_name(base, colors[:1], size_token, gender_token)
    if len(candidate) > maxlen and (size_token or gender_token):
        candidate = _build_name(base, [], size_token, gender_token)
    if len(candidate) > maxlen:
        candidate = _safe_truncate_words(candidate, maxlen)

    # 9) Nếu quá ngắn (<minlen): bổ sung 'sport' (nếu có trong tên gốc)
    if len(candidate) < minlen and had_sport:
        base_lower = base.lower()
        if "sport" not in base_lower:
            base2 = (base + " sport").strip() if base else "sport"
        else:
            base2 = base
        tmp = _build_name(base2, colors, size_token, gender_token)
        if len(tmp) <= maxlen:
            candidate = tmp
        else:
            # Nếu tràn, cắt base2 để vừa tail
            tail2 = " ".join([p for p in [" ".join(colors) if colors else "", size_token or "", gender_token or "" ] if p]).strip()
            reserved2 = len(tail2) + (1 if tail2 and base2 else 0)
            base_budget2 = maxlen - reserved2
            base2_cut = _safe_truncate_words(base2, base_budget2) if base_budget2 > 0 else ""
            candidate = _build_name(base2_cut, colors, size_token, gender_token)

    return _normalize_spaces(candidate)

df_products_template["name_simple"] = df_products_template["name"].apply(simplify_name)

In [16]:
df_products_template.to_excel("products_template.xlsx")